In [2]:
import sys
print(sys.executable)

c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\.venv\Scripts\python.exe


In [3]:
import statsmodels

print(statsmodels.__version__)

0.14.6


In [2]:
# ============================================================
# CREDRESOLVE — COLLECTIONS RECOVERY ANALYTICS
# 06 — STATISTICAL ANALYSIS
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Test statistical associations between recovery outcomes and
# collection / portfolio exposure variables.
#
# AUTHORITATIVE RECOVERY DEFINITION
# ------------------------------------------------------------
# recovered_flag:
#     1 if account has at least one SUCCESS payment
#     0 otherwise
#
# recovery amount:
#     SUM(amount) for SUCCESS payments ONLY
#     after payment_id deduplication.
#
# FAILED / PENDING / REVERSED payments are NOT recovery.
#
# IMPORTANT:
# Association does NOT imply causation.
# ============================================================


from pathlib import Path
import pandas as pd
import numpy as np
import warnings

from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

GOLDEN_DIR = PROJECT_ROOT / "data" / "golden"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("CREDRESOLVE — STATISTICAL ANALYSIS")
print("=" * 90)


# ============================================================
# 2. LOAD GOLDEN DATASETS
# ============================================================

golden_files = sorted(
    GOLDEN_DIR.glob("*_golden.csv")
)

golden = {
    file.stem.replace("_golden", ""):
    pd.read_csv(file)
    for file in golden_files
}

print(
    "Golden datasets loaded:",
    len(golden)
)


required = [
    "accounts",
    "payments",
    "calls",
    "call_attempts",
    "call_dispositions",
    "daily_targeting",
    "campaigns"
]

missing = [
    x
    for x in required
    if x not in golden
]

if missing:
    raise ValueError(
        f"Missing Golden tables: {missing}"
    )


# ============================================================
# 3. LOAD DATASETS
# ============================================================

accounts = golden["accounts"].copy()

payments = golden["payments"].copy()

calls = golden["calls"].copy()

attempts = golden["call_attempts"].copy()

dispositions = golden["call_dispositions"].copy()

targeting = golden["daily_targeting"].copy()


# ============================================================
# 4. PAYMENT RECOVERY OUTCOME
# ============================================================
#
# AUTHORITATIVE DEFINITION
# ------------------------------------------------------------
# 1. Deduplicate payment_id.
# 2. SUCCESS is the only recovery status.
# 3. recovered_flag = at least one SUCCESS payment.
# 4. recovery amount = SUCCESS amount only.
#
# This prevents duplicate payment records and non-success
# statuses from inflating recovery.
# ============================================================

payments["amount"] = pd.to_numeric(
    payments["amount"],
    errors="coerce"
).fillna(0)


payments["payment_status"] = (
    payments["payment_status"]
    .astype("string")
    .str.upper()
    .str.strip()
)


# ------------------------------------------------------------
# CHECK REQUIRED PAYMENT COLUMNS
# ------------------------------------------------------------

required_payment_columns = [
    "payment_id",
    "account_id",
    "payment_status",
    "amount"
]

missing_payment_columns = [
    column
    for column in required_payment_columns
    if column not in payments.columns
]

if missing_payment_columns:

    raise ValueError(
        "Missing payment columns: "
        f"{missing_payment_columns}"
    )


# ------------------------------------------------------------
# DEDUPLICATE PAYMENT IDs
# ------------------------------------------------------------

print("\nPayment rows before deduplication:")

print(
    len(payments)
)


if "event_at" in payments.columns:

    payments["event_at"] = pd.to_datetime(
        payments["event_at"],
        errors="coerce"
    )

    payments = (
        payments
        .sort_values(
            ["payment_id", "event_at"],
            na_position="last"
        )
        .drop_duplicates(
            subset=["payment_id"],
            keep="first"
        )
    )

else:

    payments = (
        payments
        .drop_duplicates(
            subset=["payment_id"],
            keep="first"
        )
    )


print(
    "Payment rows after payment_id deduplication:"
)

print(
    len(payments)
)


# ------------------------------------------------------------
# SUCCESS ONLY
# ------------------------------------------------------------

payments["successful_payment"] = (
    payments["payment_status"] == "SUCCESS"
)


# ------------------------------------------------------------
# SUCCESS-ONLY RECOVERY AMOUNT
# ------------------------------------------------------------

payments["successful_payment_amount"] = np.where(
    payments["successful_payment"],
    payments["amount"],
    0.0
)


# ============================================================
# 5. ACCOUNT-LEVEL PAYMENT METRICS
# ============================================================

payment_metrics = (

    payments

    .groupby("account_id")

    .agg(

        successful_payments=(
            "successful_payment",
            "sum"
        ),

        total_payment_amount=(
            "successful_payment_amount",
            "sum"
        ),

        payment_events=(
            "payment_id",
            "nunique"
        )

    )

    .reset_index()
)


# ------------------------------------------------------------
# RECOVERED ACCOUNT
# ------------------------------------------------------------

payment_metrics["recovered_flag"] = (

    payment_metrics[
        "successful_payments"
    ] > 0

).astype(int)


# ============================================================
# 6. CALL EXPOSURE
# ============================================================

if "duration_sec" in calls.columns:

    calls["duration_sec"] = pd.to_numeric(
        calls["duration_sec"],
        errors="coerce"
    ).fillna(0)

else:

    calls["duration_sec"] = 0


call_metrics = (

    calls

    .groupby("account_id")

    .agg(

        total_calls=(
            "call_id",
            "nunique"
        ),

        total_call_duration_sec=(
            "duration_sec",
            "sum"
        ),

        unique_agents=(
            "agent_id",
            "nunique"
        ),

        unique_campaigns=(
            "campaign_id",
            "nunique"
        ),

        unique_vendors=(
            "vendor_id",
            "nunique"
        )

    )

    .reset_index()
)


# ============================================================
# 7. CALL ATTEMPTS
# ============================================================

attempt_metrics = (

    attempts

    .groupby("account_id")

    .agg(

        total_attempts=(
            "attempt_id",
            "nunique"
        ),

        max_attempt_number=(
            "attempt_no",
            "max"
        )

    )

    .reset_index()
)


# ============================================================
# 8. CALL DISPOSITIONS
# ============================================================

disposition_metrics = (

    dispositions

    .groupby("account_id")

    .agg(

        disposition_events=(
            "disposition_id",
            "nunique"
        ),

        disposition_codes=(
            "disposition_code",
            "nunique"
        )

    )

    .reset_index()
)


# ============================================================
# 9. TARGETING
# ============================================================

# Support datasets where the targeting identifier may differ.

if "target_id" in targeting.columns:

    targeting_id = "target_id"

elif "targeting_id" in targeting.columns:

    targeting_id = "targeting_id"

else:

    targeting_id = None


targeting_agg = {

    "targeting_events":
        (
            targeting_id,
            "nunique"
        )
        if targeting_id
        else
        (
            "account_id",
            "size"
        )
}


if "campaign_id" in targeting.columns:

    targeting_agg[
        "campaigns_targeted"
    ] = (
        "campaign_id",
        "nunique"
    )


targeting_metrics = (

    targeting

    .groupby("account_id")

    .agg(**targeting_agg)

    .reset_index()
)


# ============================================================
# 10. BUILD ACCOUNT-LEVEL DATASET
# ============================================================

analysis = accounts.merge(
    payment_metrics,
    on="account_id",
    how="left"
)


analysis = analysis.merge(
    call_metrics,
    on="account_id",
    how="left"
)


analysis = analysis.merge(
    attempt_metrics,
    on="account_id",
    how="left"
)


analysis = analysis.merge(
    disposition_metrics,
    on="account_id",
    how="left"
)


analysis = analysis.merge(
    targeting_metrics,
    on="account_id",
    how="left"
)


# ============================================================
# 11. FILL MISSING VALUES
# ============================================================

numeric_columns = [

    "successful_payments",

    "total_payment_amount",

    "payment_events",

    "total_calls",

    "total_call_duration_sec",

    "unique_agents",

    "unique_campaigns",

    "unique_vendors",

    "total_attempts",

    "max_attempt_number",

    "disposition_events",

    "disposition_codes",

    "targeting_events",

    "campaigns_targeted"

]


for column in numeric_columns:

    if column in analysis.columns:

        analysis[column] = pd.to_numeric(
            analysis[column],
            errors="coerce"
        ).fillna(0)


analysis["recovered_flag"] = (
    analysis["recovered_flag"]
    .fillna(0)
    .astype(int)
)


# ============================================================
# 12. DPD BUCKET
# ============================================================

def dpd_bucket(value):

    if pd.isna(value):
        return "Missing"

    if value <= 0:
        return "Current"

    if value <= 30:
        return "1-30"

    if value <= 60:
        return "31-60"

    if value <= 90:
        return "61-90"

    return "90+"


analysis["dpd_bucket"] = (
    analysis["dpd"]
    .apply(dpd_bucket)
)


# ============================================================
# 13. OVERALL RECOVERY
# ============================================================

n_accounts = int(
    analysis["account_id"].nunique()
)


n_recovered = int(
    analysis["recovered_flag"].sum()
)


overall_rate = (

    n_recovered / n_accounts

    if n_accounts > 0

    else np.nan
)


# IMPORTANT:
# This is already SUCCESS-only and payment_id-deduplicated.

total_recovery = float(
    analysis[
        "total_payment_amount"
    ].sum()
)


overall = pd.DataFrame([{

    "accounts":
        n_accounts,

    "recovered_accounts":
        n_recovered,

    "recovery_rate":
        overall_rate,

    "total_recovery_amount":
        total_recovery

}])


print("\n" + "=" * 90)
print("OVERALL RECOVERY STATISTICS")
print("=" * 90)

print(
    overall.to_string(
        index=False
    )
)


overall.to_csv(
    OUTPUT_DIR /
    "statistical_overall_recovery.csv",
    index=False
)


# ============================================================
# 14. CATEGORICAL DRIVER TESTS
# ============================================================

print("\n" + "=" * 90)
print("CATEGORICAL DRIVER TESTS")
print("=" * 90)


categorical_drivers = [

    "loan_type",
    "risk_segment",
    "status",
    "dpd_bucket"

]


chi_results = []


for driver in categorical_drivers:

    if driver not in analysis.columns:
        continue


    table = pd.crosstab(
        analysis[driver],
        analysis["recovered_flag"]
    )


    if (
        table.shape[0] >= 2
        and table.shape[1] >= 2
    ):

        chi2, p_value, dof, expected = (
            stats.chi2_contingency(table)
        )

    else:

        chi2 = np.nan
        p_value = np.nan
        dof = np.nan


    chi_results.append({

        "driver":
            driver,

        "chi_square":
            chi2,

        "degrees_of_freedom":
            dof,

        "p_value":
            p_value

    })


chi_df = pd.DataFrame(
    chi_results
)


chi_df["significant_5pct"] = (
    chi_df["p_value"] < 0.05
)


print(
    chi_df.to_string(
        index=False
    )
)


chi_df.to_csv(
    OUTPUT_DIR /
    "statistical_categorical_tests.csv",
    index=False
)


# ============================================================
# 15. NUMERIC DRIVER TESTS
# ============================================================

print("\n" + "=" * 90)
print("NUMERIC DRIVER TESTS")
print("=" * 90)


numeric_drivers = [

    "dpd",
    "principal_amount",
    "outstanding_amount",
    "total_calls",
    "total_call_duration_sec",
    "total_attempts",
    "targeting_events",
    "campaigns_targeted",
    "disposition_events",
    "unique_agents",
    "unique_vendors"

]


numeric_results = []


for driver in numeric_drivers:

    if driver not in analysis.columns:
        continue


    recovered = (
        analysis.loc[
            analysis["recovered_flag"] == 1,
            driver
        ]
        .dropna()
    )


    not_recovered = (
        analysis.loc[
            analysis["recovered_flag"] == 0,
            driver
        ]
        .dropna()
    )


    if (
        len(recovered) > 1
        and len(not_recovered) > 1
    ):

        statistic, p_value = (
            stats.mannwhitneyu(
                recovered,
                not_recovered,
                alternative="two-sided"
            )
        )

        recovered_median = (
            recovered.median()
        )

        not_recovered_median = (
            not_recovered.median()
        )

    else:

        statistic = np.nan
        p_value = np.nan
        recovered_median = np.nan
        not_recovered_median = np.nan


    numeric_results.append({

        "driver":
            driver,

        "recovered_median":
            recovered_median,

        "not_recovered_median":
            not_recovered_median,

        "median_difference":
            recovered_median
            - not_recovered_median,

        "mann_whitney_u":
            statistic,

        "p_value":
            p_value

    })


numeric_df = pd.DataFrame(
    numeric_results
)


numeric_df["significant_5pct"] = (
    numeric_df["p_value"] < 0.05
)


print(
    numeric_df.to_string(
        index=False
    )
)


numeric_df.to_csv(
    OUTPUT_DIR /
    "statistical_numeric_tests.csv",
    index=False
)


# ============================================================
# 16. CLIFF'S DELTA
# ============================================================

print("\n" + "=" * 90)
print("EFFECT SIZE ANALYSIS")
print("=" * 90)


def cliffs_delta(x, y):

    x = np.asarray(x)
    y = np.asarray(y)

    x = x[
        ~pd.isna(x)
    ]

    y = y[
        ~pd.isna(y)
    ]

    if (
        len(x) == 0
        or len(y) == 0
    ):
        return np.nan


    greater = (
        x[:, None] >
        y[None, :]
    ).sum()


    smaller = (
        x[:, None] <
        y[None, :]
    ).sum()


    return (
        greater - smaller
    ) / (
        len(x) * len(y)
    )


effect_results = []


for driver in numeric_drivers:

    if driver not in analysis.columns:
        continue


    recovered = (
        analysis.loc[
            analysis["recovered_flag"] == 1,
            driver
        ]
        .dropna()
        .values
    )


    not_recovered = (
        analysis.loc[
            analysis["recovered_flag"] == 0,
            driver
        ]
        .dropna()
        .values
    )


    effect = cliffs_delta(
        recovered,
        not_recovered
    )


    effect_results.append({

        "driver":
            driver,

        "cliffs_delta":
            effect,

        "absolute_cliffs_delta":
            abs(effect)
            if pd.notna(effect)
            else np.nan

    })


effect_df = pd.DataFrame(
    effect_results
)


print(
    effect_df.to_string(
        index=False
    )
)


effect_df.to_csv(
    OUTPUT_DIR /
    "statistical_effect_sizes.csv",
    index=False
)


# ============================================================
# 17. MULTIPLE-TESTING CORRECTION
# ============================================================

all_pvalues = pd.concat([

    chi_df["p_value"],

    numeric_df["p_value"]

]).dropna()


number_of_tests = len(
    all_pvalues
)


bonferroni_alpha = (

    0.05 / number_of_tests

    if number_of_tests > 0

    else np.nan
)


chi_df[
    "bonferroni_significant"
] = (
    chi_df["p_value"]
    < bonferroni_alpha
)


numeric_df[
    "bonferroni_significant"
] = (
    numeric_df["p_value"]
    < bonferroni_alpha
)


print(
    "\nNumber of statistical tests:",
    number_of_tests
)


print(
    "Bonferroni alpha:",
    bonferroni_alpha
)


# ============================================================
# 18. LOGISTIC REGRESSION
# ============================================================
#
# Outcome:
#     recovered_flag
#
# recovered_flag is SUCCESS-payment based.
#
# Interpretation:
#     ASSOCIATION ONLY.
#
# No causal claim.
# ============================================================

print("\n" + "=" * 90)
print("MULTIVARIABLE LOGISTIC REGRESSION")
print("=" * 90)


model_columns = [

    "recovered_flag",

    "dpd",

    "principal_amount",

    "outstanding_amount",

    "total_calls",

    "total_attempts",

    "targeting_events",

    "loan_type",

    "risk_segment"

]


model_columns = [
    column
    for column in model_columns
    if column in analysis.columns
]


model_data = analysis[
    model_columns
].copy()


for column in [

    "dpd",

    "principal_amount",

    "outstanding_amount",

    "total_calls",

    "total_attempts",

    "targeting_events"

]:

    if column in model_data.columns:

        model_data[column] = pd.to_numeric(
            model_data[column],
            errors="coerce"
        )


model_data = model_data.dropna()


for column in [

    "principal_amount",

    "outstanding_amount"

]:

    if column in model_data.columns:

        low = model_data[
            column
        ].quantile(0.01)

        high = model_data[
            column
        ].quantile(0.99)

        model_data[column] = (
            model_data[column]
            .clip(low, high)
        )


formula = (

    "recovered_flag ~ "

    "dpd + "

    "principal_amount + "

    "outstanding_amount + "

    "total_calls + "

    "total_attempts + "

    "targeting_events + "

    "C(loan_type) + "

    "C(risk_segment)"

)


try:

    model = smf.logit(
        formula=formula,
        data=model_data
    ).fit(
        disp=False,
        maxiter=200
    )


    coefficients = pd.DataFrame({

        "variable":
            model.params.index,

        "coefficient":
            model.params.values,

        "odds_ratio":
            np.exp(
                model.params.values
            ),

        "p_value":
            model.pvalues.values,

        "ci_lower_odds_ratio":
            np.exp(
                model.conf_int()[0].values
            ),

        "ci_upper_odds_ratio":
            np.exp(
                model.conf_int()[1].values
            )

    })


    coefficients[
        "significant_5pct"
    ] = (
        coefficients["p_value"]
        < 0.05
    )


    print(
        coefficients.to_string(
            index=False
        )
    )


    coefficients.to_csv(
        OUTPUT_DIR /
        "statistical_logistic_regression.csv",
        index=False
    )


    model_fit = pd.DataFrame([{

        "observations":
            int(model.nobs),

        "pseudo_r_squared":
            model.prsquared,

        "aic":
            model.aic,

        "bic":
            model.bic

    }])


    print("\nMODEL FIT")

    print(
        model_fit.to_string(
            index=False
        )
    )


    model_fit.to_csv(
        OUTPUT_DIR /
        "statistical_logistic_model_fit.csv",
        index=False
    )


except Exception as error:

    print(
        "Logistic regression failed:"
    )

    print(error)


# ============================================================
# 19. CALL EXPOSURE ANALYSIS
# ============================================================

print("\n" + "=" * 90)
print("CALL EXPOSURE ANALYSIS")
print("=" * 90)


analysis[
    "call_exposure_group"
] = pd.cut(

    analysis["total_calls"],

    bins=[
        -np.inf,
        2,
        3,
        4,
        np.inf
    ],

    labels=[
        "0-2",
        "3",
        "4",
        "5+"
    ]

)


exposure_summary = (

    analysis

    .groupby(
        "call_exposure_group",
        observed=False
    )

    .agg(

        accounts=(
            "account_id",
            "nunique"
        ),

        recovered_accounts=(
            "recovered_flag",
            "sum"
        ),

        recovery_amount=(
            "total_payment_amount",
            "sum"
        ),

        avg_dpd=(
            "dpd",
            "mean"
        )

    )

    .reset_index()
)


exposure_summary[
    "recovery_rate"
] = (

    exposure_summary[
        "recovered_accounts"
    ]

    /

    exposure_summary[
        "accounts"
    ]

)


print(
    exposure_summary.to_string(
        index=False
    )
)


exposure_summary.to_csv(
    OUTPUT_DIR /
    "statistical_call_exposure.csv",
    index=False
)


# ============================================================
# 20. STATISTICAL EVIDENCE SUMMARY
# ============================================================

evidence_rows = []


for _, row in chi_df.iterrows():

    evidence_rows.append({

        "analysis":
            "Chi-square",

        "driver":
            row["driver"],

        "p_value":
            row["p_value"],

        "significant_5pct":
            row["significant_5pct"],

        "bonferroni_significant":
            row["bonferroni_significant"]

    })


for _, row in numeric_df.iterrows():

    evidence_rows.append({

        "analysis":
            "Mann-Whitney U",

        "driver":
            row["driver"],

        "p_value":
            row["p_value"],

        "significant_5pct":
            row["significant_5pct"],

        "bonferroni_significant":
            row["bonferroni_significant"]

    })


evidence_df = pd.DataFrame(
    evidence_rows
)


print("\n" + "=" * 90)
print("STATISTICAL EVIDENCE SUMMARY")
print("=" * 90)


print(
    evidence_df.to_string(
        index=False
    )
)


evidence_df.to_csv(
    OUTPUT_DIR /
    "statistical_evidence_summary.csv",
    index=False
)


# ============================================================
# 21. FINAL SUMMARY
# ============================================================

significant_5pct = int(

    evidence_df[
        "significant_5pct"
    ]
    .fillna(False)
    .sum()

)


significant_bonferroni = int(

    evidence_df[
        "bonferroni_significant"
    ]
    .fillna(False)
    .sum()

)


final_summary = pd.DataFrame([{

    "accounts_analyzed":
        n_accounts,

    "recovered_accounts":
        n_recovered,

    "overall_recovery_rate":
        overall_rate,

    "total_recovery_amount":
        total_recovery,

    "statistical_tests_run":
        number_of_tests,

    "significant_at_5pct":
        significant_5pct,

    "significant_after_bonferroni":
        significant_bonferroni,

    "causal_claims_made":
        False

}])


print("\n" + "=" * 90)
print("STATISTICAL ANALYSIS SUMMARY")
print("=" * 90)


print(
    final_summary.to_string(
        index=False
    )
)


final_summary.to_csv(
    OUTPUT_DIR /
    "statistical_analysis_summary.csv",
    index=False
)


# ============================================================
# 22. SAVE ACCOUNT-LEVEL DATASET
# ============================================================

analysis.to_csv(
    OUTPUT_DIR /
    "statistical_account_level_dataset.csv",
    index=False
)


# ============================================================
# 23. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("SUCCESS-ONLY RECOVERY VALIDATION")
print("=" * 90)

print(
    "Accounts:",
    n_accounts
)

print(
    "Recovered accounts:",
    n_recovered
)

print(
    "Recovery rate:",
    round(
        overall_rate * 100,
        2
    ),
    "%"
)

print(
    "SUCCESS-only recovery amount:",
    round(
        total_recovery,
        2
    )
)


# ============================================================
# 24. FINAL STATUS
# ============================================================

print("\n" + "=" * 90)
print("STATISTICAL ANALYSIS COMPLETE")
print("=" * 90)

print(
    "Recovery definition: SUCCESS payments only"
)

print(
    "Payment IDs deduplicated: TRUE"
)

print(
    "Recovery amount: SUCCESS payments only"
)

print(
    "Raw source data modified: FALSE"
)

print(
    "Causal claims made: FALSE"
)

print(
    "Statistical associations tested: TRUE"
)

print(
    "Multiple-testing correction: TRUE"
)

print(
    f"Outputs saved to: {OUTPUT_DIR}"
)

CREDRESOLVE — STATISTICAL ANALYSIS
Golden datasets loaded: 18

Payment rows before deduplication:
25500
Payment rows after payment_id deduplication:
25000

OVERALL RECOVERY STATISTICS
 accounts  recovered_accounts  recovery_rate  total_recovery_amount
    30000               13284         0.4428           1.315584e+09

CATEGORICAL DRIVER TESTS
      driver  chi_square  degrees_of_freedom  p_value  significant_5pct
   loan_type    4.101268                   4 0.392474             False
risk_segment    2.678277                   3 0.443932             False
      status    1.155898                   3 0.763600             False
  dpd_bucket   10.887965                   4 0.027852              True

NUMERIC DRIVER TESTS
                 driver  recovered_median  not_recovered_median  median_difference  mann_whitney_u  p_value  significant_5pct
                    dpd            45.000                 45.00              0.000     111030410.0 0.997056             False
       principal_amo